In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import decoupler as dc
import os
import gseapy
import matplotlib.pyplot as plt
import squidpy as sq
import pickle
import anndata as ad
import re
from glob import glob
import copy

adata_infile = "adata.h5ad"


adata = sc.read_h5ad(adata_infile)

mapping_file = os.path.expanduser(
    "mapping.csv"
)

# ----------------------------
# 2️⃣ Load mapping file
# ----------------------------
mapping_df = pd.read_csv(mapping_file, dtype=str)
mapping_df = mapping_df.loc[:, ~mapping_df.columns.duplicated()]

IGNORE_PREFIX_N = 0

cell_type_colors = {
    # --- Malignant epithelial ---
    'CEACAM-high tumor epithelial cells': '#6BA4F8',   # softened azure blue
    'Cycling Tumor Cells': '#F4BA63',                 # softened amber
    'Mucin-producing tumor cells': '#E59973',         # softened coral
    'Inflamed primary tumor epithelial cells': '#FF6259', # softened red

    # --- Tumor microenvironment ---
    'Complement immunosuppressive macrophages (TAMs)': '#874284',  # softened purple
    'Systemic inflammatory macrophage program (TAMs)': '#61385B',   # softened plum
    'CAFs (Cancer associated fibroblasts)': '#9A5766',              # softened burgundy
    'Pericyte-enriched endothelial cells': '#4F709D',              # softened navy

    # --- Immune cells ---
    'Cytotoxic T cells': '#998CFA',   # softened lavender
    'Plasma Cells': '#56B356',        # softened green
}



sc.pl.umap(
    adata,
    color='cell_type',
    palette=cell_type_colors,
    frameon=False,
)


In [ ]:
import os
import scanpy as sc
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib
matplotlib.use("Agg")  
import matplotlib.pyplot as plt

# =========================================================
# OUTPUT DIR 
# =========================================================
OUTDIR = "docs/plots"
os.makedirs(OUTDIR, exist_ok=True)

# =========================================================
# 0. CELL TYPE DEFINITIONS
# =========================================================

celltype_order = [
    'Mucin-producing tumor cells',
    'CEACAM-high tumor epithelial cells',
    'Cycling Tumor Cells',
    'Inflamed primary tumor epithelial cells',
    'Complement immunosuppressive macrophages (TAMs)',
    'Systemic inflammatory macrophage program (TAMs)',
    'CAFs (Cancer associated fibroblasts)',
    'Pericyte-enriched endothelial cells',
    'Cytotoxic T cells',
    'B lymphocytes',
    'Plasma Cells'
]

short_labels = {
    'CEACAM-high tumor epithelial cells': 'CEACAM+ Tumor',
    'Cycling Tumor Cells': 'Cycling Tumor',
    'Mucin-producing tumor cells': 'Mucin Tumor',
    'Inflamed primary tumor epithelial cells': 'Inflamed Tumor',
    'Complement immunosuppressive macrophages (TAMs)': 'Complement TAMs',
    'Systemic inflammatory macrophage program (TAMs)': 'Inflammatory TAMs',
    'CAFs (Cancer associated fibroblasts)': 'CAFs',
    'Pericyte-enriched endothelial cells': 'Pericyte ECs',
    'Cytotoxic T cells': 'Cytotoxic T',
    'B lymphocytes': 'B cells',
    'Plasma Cells': 'Plasma'
}

# =========================================================
# LOAD DATA 
# =========================================================
def load_data(path="adata.h5ad"):
    return sc.read_h5ad(path)

# =========================================================
# UTIL
# =========================================================
def pct_expr(adata, gene, group):
    in_g = adata[adata.obs['cell_type'] == group]
    out_g = adata[adata.obs['cell_type'] != group]

    pct_in = np.mean(in_g[:, gene].X > 0)
    pct_out = np.mean(out_g[:, gene].X > 0)

    return pct_in, pct_out

# =========================================================
# MAIN PIPELINE
# =========================================================
def main(adata):

    # =====================================================
    # FILTER
    # =====================================================
    exclude_cts = [
        'Inflamed primary tumor epithelial cells',
        'Systemic inflammatory macrophage program (TAMs)'
    ]

    adata = adata[~adata.obs['cell_type'].isin(exclude_cts)].copy()

    min_cells = 100
    keep_cts = adata.obs['cell_type'].value_counts()
    keep_cts = keep_cts[keep_cts >= min_cells].index
    adata = adata[adata.obs['cell_type'].isin(keep_cts)].copy()

    # =====================================================
    # ORDER
    # =====================================================
    valid_order = [ct for ct in celltype_order if ct in adata.obs['cell_type'].unique()]

    adata.obs['cell_type'] = pd.Categorical(
        adata.obs['cell_type'],
        categories=valid_order,
        ordered=True
    )

    adata.obs['cell_type_short'] = adata.obs['cell_type'].map(short_labels)

    short_order = [short_labels[c] for c in valid_order if c in short_labels]

    adata.obs['cell_type_short'] = pd.Categorical(
        adata.obs['cell_type_short'],
        categories=short_order,
        ordered=True
    )

    # =====================================================
    # SPLIT
    # =====================================================
    tumor_cts = [
        'Mucin-producing tumor cells',
        'CEACAM-high tumor epithelial cells',
        'Cycling Tumor Cells'
    ]

    adata_tumor = adata[adata.obs['cell_type'].isin(tumor_cts)].copy()
    adata_non_tumor = adata[~adata.obs['cell_type'].isin(tumor_cts)].copy()

    # =====================================================
    # MARKERS (NON-TUMOR)
    # =====================================================
    sc.tl.rank_genes_groups(adata_non_tumor, groupby="cell_type", method="wilcoxon")
    ranked_non = sc.get.rank_genes_groups_df(adata_non_tumor, None)

    filtered_non = ranked_non[
        (ranked_non['logfoldchanges'] > 1) &
        (ranked_non['pvals_adj'] < 0.01)
    ]

    non_tumor_markers = []
    for _, row in filtered_non.iterrows():
        try:
            pct_in, pct_out = pct_expr(adata_non_tumor, row['names'], row['group'])
            if pct_in > 0.2 and pct_out < 0.2:
                non_tumor_markers.append(row)
        except:
            continue

    non_tumor_df = pd.DataFrame(non_tumor_markers)

    # =====================================================
    # MARKERS (TUMOR)
    # =====================================================
    sc.tl.rank_genes_groups(adata_tumor, groupby="cell_type", method="wilcoxon")
    ranked_tumor = sc.get.rank_genes_groups_df(adata_tumor, None)

    filtered_tumor = ranked_tumor[
        (ranked_tumor['logfoldchanges'] > 0.3) &
        (ranked_tumor['pvals_adj'] < 0.05)
    ]

    tumor_markers = []
    for _, row in filtered_tumor.iterrows():
        try:
            pct_in, pct_out = pct_expr(adata_tumor, row['names'], row['group'])
            if pct_in > 0.1 and (pct_in - pct_out) > 0.1:
                tumor_markers.append(row)
        except:
            continue

    tumor_df = pd.DataFrame(tumor_markers)

    # =====================================================
    # TOP GENES
    # =====================================================
    top_n = 5

    non_tumor_top = non_tumor_df.sort_values(
        ['group', 'logfoldchanges'], ascending=[True, False]
    ).groupby('group').head(top_n)

    tumor_top = tumor_df.sort_values(
        ['group', 'logfoldchanges'], ascending=[True, False]
    ).groupby('group').head(top_n)

    combined = pd.concat([non_tumor_top, tumor_top])

    marker_dict = combined.groupby('group')['names'].apply(list).to_dict()

    ordered_genes = []
    for ct in valid_order:
        if ct in marker_dict:
            ordered_genes.extend(marker_dict[ct])

    ordered_genes = list(dict.fromkeys(ordered_genes))  # dedupe

    # =====================================================
    # HEATMAP (SCANPY)
    # =====================================================
    sc.set_figure_params(figsize=(14, 8), fontsize=10)

    sc.pl.heatmap(
        adata,
        var_names=ordered_genes,
        groupby="cell_type_short",
        standard_scale="var",
        cmap="viridis",
        dendrogram=False,
        show_gene_labels=True,
        save=None,
        show=False
    )

    # =====================================================
    # EXPORT MATRIX FOR WEB
    # =====================================================
    filtered_genes = ordered_genes

    X = pd.DataFrame(
        adata[:, filtered_genes].X.toarray()
        if hasattr(adata.X, "toarray") else adata[:, filtered_genes].X,
        columns=filtered_genes,
        index=adata.obs['cell_type_short']
    )

    mean_df = X.groupby(level=0).mean().loc[short_order]
    median_df = X.groupby(level=0).median().loc[short_order]
    max_df = X.groupby(level=0).max().loc[short_order]

    mean_df.to_csv(f"{OUTDIR}/mean_expression.csv")
    median_df.to_csv(f"{OUTDIR}/median_expression.csv")
    max_df.to_csv(f"{OUTDIR}/max_expression.csv")

    # =====================================================
    # PLOTTING
    # =====================================================
    def plot_heatmap(df, title, filename):
        fig, ax = plt.subplots(figsize=(14, 7))

        sns.heatmap(df, cmap="Reds", ax=ax, vmin=1)

        ax.set_title(title, fontsize=18, pad=12)
        ax.set_xlabel("Genes", fontsize=14)
        ax.set_ylabel("Cell Types", fontsize=14)

        ax.tick_params(axis='x', rotation=90)
        ax.tick_params(axis='y', rotation=0)

        plt.tight_layout()

        fig.savefig(f"{OUTDIR}/{filename}.png", dpi=300)
        fig.savefig(f"{OUTDIR}/{filename}.svg")  # BEST for GitHub Pages
        plt.close(fig)

    plot_heatmap(mean_df, "Mean Expression", "mean_heatmap")
    plot_heatmap(median_df, "Median Expression", "median_heatmap")
    plot_heatmap(max_df, "Max Expression", "max_heatmap")


# =========================================================
# RUN
# =========================================================
if __name__ == "__main__":
    adata = load_data("adata.h5ad")
    main(adata)
